# QLoRA training on Kaggle (phase 4)

Thin wrapper (CLAUDE.md §4): installs Unsloth with pinned versions, installs the `t2sql` package from GitHub, rebuilds the dataset, then runs `t2sql.train.sft`. No logic lives here.

**Before running**
- Settings → Accelerator: **GPU T4 x2** (only the first GPU is used); Internet: **On**.
- Add Input → the private dataset holding `eco2mix.duckdb`.
- Add-ons → Secrets: `WANDB_API_KEY` and `HF_TOKEN` (write), both attached to this notebook.

**How to run**: run every cell by hand down to the short test and read what it prints. Only then *Save Version → Save & Run All* (set `SHORT_TEST = False` first if it already passed). Every 33 updates a checkpoint is saved here and pushed to a private Hub repo of the run.

**If a session is killed**: run the notebook again with `RESUME = True`: training continues from the last checkpoint on the Hub.

In [ ]:
REPO = "https://github.com/zaninip/domain-text2sql"
COMMIT = "main"  # pin a commit SHA to reproduce a run exactly
# Unsloth works with exact versions of its neighbours: the recipe of its Kaggle notebooks
UNSLOTH = "2026.9.14"
TRANSFORMERS = "4.56.2"
TRL = "0.22.2"
SHORT_TEST = True  # 2 updates, save and push, then resume to 4: checks the Hub and resuming
RESUME = False  # True only to continue a full run whose session was killed

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch and xformers are reinstalled together: xformers must be built for that exact torch
!pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install -q unsloth=={UNSLOTH}
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q transformers=={TRANSFORMERS}
!pip install -q --no-deps trl=={TRL}

In [ ]:
import subprocess

# In a fresh interpreter: this kernel must not import torch before training does
VERSIONS = """
import unsloth, torch, transformers, trl, peft, bitsandbytes, xformers
for module in (unsloth, torch, transformers, trl, peft, bitsandbytes, xformers):
    print(f"{module.__name__:13} {module.__version__}")
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
"""
_ = subprocess.run(["python", "-c", VERSIONS], check=True)

In [ ]:
!git clone -q {REPO} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout -q {COMMIT} && git log -1 --oneline
# Base dependencies only (duckdb, sqlglot, yaml): the training libraries stay as installed above
!pip install -q -e .

In [ ]:
import glob
import shutil
from pathlib import Path

found = glob.glob("/kaggle/input/**/eco2mix.duckdb", recursive=True)
assert found, "attach the dataset holding eco2mix.duckdb (Add Input)"
Path("data").mkdir(exist_ok=True)
shutil.copy(found[0], "data/eco2mix.duckdb")

In [ ]:
# Same command as `make dataset`; expected: train 2126 / val 402 / test 490
!python -u -m t2sql.dataset.build

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()  # the keys are read here and never printed
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_PROJECT"] = "domain-text2sql"
# Two T4s visible would make the trainer split each batch across both (DataParallel),
# which Unsloth does not support: one GPU only
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


def train(*extra: str) -> None:
    """One run of t2sql.train.sft; check=True stops the notebook if it fails."""
    command = ["python", "-u", "-m", "t2sql.train.sft", *extra]
    subprocess.run(command, check=True)

**Short test** (~15 min, no validation loss): 2 updates with a checkpoint pushed to the Hub, then a second launch with `--resume` that must start from update 2, not 0, and stop at 4. Read the lines `base weights loaded from`, `trainable parameters` (17,432,576), `first batch` (about 74 of 4,650 tokens with loss), `checkpoints every 2 updates, pushed to ...-smoke` and `resuming from`; on the Hub, the private repo `zaninip/eco2mix-sql-...-smoke` must hold a `last-checkpoint` folder.

In [ ]:
if SHORT_TEST:
    short = ["--save-steps", "2", "--no-eval"]
    train("--max-steps", "2", *short)
    train("--max-steps", "4", *short, "--resume")

**Full run**: one epoch, 133 updates, about 3.5 h with the validation loss every 33 updates. Checkpoints `checkpoint-33`, `-66`, `-99`, `-132` stay in this notebook's output (evaluated by the evaluation notebook) and are pushed to the run's Hub repo.

In [ ]:
train(*(["--resume"] if RESUME else []))